In [0]:
from pyspark.sql import functions as F

#load silver tables
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")
cpih = spark.table("scottish_housing_ws.silver.cpih")

In [0]:
#all items CPIH
cpih_all = (
    cpih
    .filter(F.col("component_code")=="CP00")
    .select("year_month", "index_value")
)

#find index value for reference month
reference_row = (
    cpih_all
    .orderBy(
        F.when(F.col("year_month")== "2015-01",0).otherwise(1),
        F.col("year_month")
    )
    .limit(1)
    .collect()
)

reference_index = reference_row[0]["index_value"]
reference_month = reference_row[0]["year_month"]

print(f"CPIH rebase reference month: {reference_month} with index value: {reference_index}")

cpih_rebased = (
    cpih_all
    .withColumn(
        "cpih_rebased",
        F.col("index_value")/F.lit(reference_index)*F.lit(100)
    )
    .select("year_month", "cpih_rebased")
)
hpi.printSchema()
cpih.printSchema()

In [0]:
gold = (
    hpi
    .join(cpih_rebased, on="year_month", how="inner")
    .withColumn(
        "real_average_price",
        F.col("average_price")/F.col("cpih_rebased")
    )
    .withColumn(
        "real_index",
        F.round(F.col("average_price")/(F.col("cpih_rebased")/F.lit(100)),2)
    )
    .select(
        "report_date",
        "year_month",
        "region_name",
        "area_code",
        F.col("average_price").alias("nominal_average_price"),
        "real_average_price",
        "cpih_rebased",
        "sales_volume"
    )
    .orderBy("area_code","report_date")
)

In [0]:
print(f"Row count: {gold.count()}")
gold.filter(F.col("area_code") == "S92000003").orderBy("report_date").show(5)
gold.filter(F.col("area_code") == "S92000003").orderBy(F.col("report_date").desc()).show(5)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.nominal_vs_real_house_prices")
)

In [0]:

(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_nominal_vs_real_house_prices/")
)

print("Gold 01 complete.")

In [0]:
#house price index vs Consumer Price Index including Housing
#have scottish house prices actually risen in real terms, or has some of the nominal price increase just been inflation?
